In [ ]:
# ==================== PARAMETERS ====================

# Đường dẫn dữ liệu đã feature engineering từ notebook 03
INPUT_DATA_PATH = "data/processed/feature_engineered_data.parquet"

# Thư mục lưu kết quả
OUTPUT_DIR = "data/processed"

# Tên file output
ARIMA_RESULTS_FILENAME = "arima_results.csv"
ARIMA_METRICS_FILENAME = "arima_metrics.csv"

# Cột mục tiêu (cần dự báo)
TARGET_COLUMN = "Global_active_power"

# Tỷ lệ tập test (20% dữ liệu cuối)
TEST_SIZE = 0.2

# Tham số ARIMA - (p,d,q)
ARIMA_ORDER = (1, 1, 1)

# Tham số Seasonal - (P,D,Q,m) với m=24 (chu kỳ 24h)
SEASONAL_ORDER = (0, 0, 0, 24)

# Bật/tắt các biểu đồ
PLOT_ARIMA_FORECAST = True
PLOT_RESIDUALS = True

In [ ]:
# ==================== SETUP ====================
%load_ext autoreload
%autoreload 2

import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Xác định project_root linh hoạt
cwd = os.getcwd()
if os.path.basename(cwd) == "notebooks":
    project_root = os.path.abspath("..")
else:
    project_root = cwd

src_path = os.path.join(project_root, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

# Import thư viện tự viết - SimpleARIMA
try:
    from src.simple_arima import SimpleARIMA, calculate_mae, calculate_rmse, calculate_smape
    SIMPLE_ARIMA_AVAILABLE = True
    print("✅ SimpleARIMA đã sẵn sàng!")
except ImportError as e:
    print(f"⚠️ Lỗi import SimpleARIMA: {e}")
    print("   Vui lòng đảm bảo file src/simple_arima.py tồn tại")
    SIMPLE_ARIMA_AVAILABLE = False

# Import thư viện tự viết khác
from src.energy_forecast_library import DataLoader, Utils

# Tạo thư mục output nếu chưa có
os.makedirs(os.path.join(project_root, OUTPUT_DIR), exist_ok=True)
os.makedirs(os.path.join(project_root, "outputs/figures"), exist_ok=True)

print(f"✅ Project root: {project_root}")
print(f"✅ Output dir: {OUTPUT_DIR}")

In [ ]:
# ==================== LOAD DATA ====================
data_path = os.path.join(project_root, INPUT_DATA_PATH)

if not os.path.exists(data_path):
    print(f"❌ Không tìm thấy file: {data_path}")
    print("Vui lòng chạy notebook 03 trước!")
else:
    df = DataLoader.load_processed_data(data_path)
    
    print(f"✅ Đã tải dữ liệu từ: {INPUT_DATA_PATH}")
    print(f"📊 Kích thước: {df.shape[0]:,} dòng, {df.shape[1]} cột")
    print(f"📅 Thời gian: {df.index.min()} → {df.index.max()}")
    print(f"📌 Cột mục tiêu: {TARGET_COLUMN}")
    
    df.head()

In [ ]:
# ==================== CHIA TRAIN/TEST ====================
print("📊 Chia dữ liệu train/test theo thời gian...")
print(f"📌 Tỷ lệ test: {TEST_SIZE*100:.0f}%")

# Chia dữ liệu theo thời gian
n = len(df)
split_idx = int(n * (1 - TEST_SIZE))

train = df.iloc[:split_idx]
test = df.iloc[split_idx:]

y_train = train[TARGET_COLUMN]
y_test = test[TARGET_COLUMN]

print(f"\n✅ Train: {len(y_train):,} samples ({train.index[0]} → {train.index[-1]})")
print(f"✅ Test: {len(y_test):,} samples ({test.index[0]} → {test.index[-1]})")

# Vẽ biểu đồ chia train/test
fig, ax = plt.subplots(figsize=(15, 5))

# Lấy mẫu để vẽ (1000 điểm cuối train + 500 điểm đầu test)
train_sample = y_train.iloc[-1000:] if len(y_train) > 1000 else y_train
test_sample = y_test.iloc[:500] if len(y_test) > 500 else y_test

ax.plot(train_sample.index, train_sample.values, color='blue', linewidth=1, label='Train')
ax.plot(test_sample.index, test_sample.values, color='red', linewidth=1, label='Test')
ax.axvline(x=y_test.index[0], color='black', linestyle='--', alpha=0.7, label='Split point')
ax.set_xlabel('Thời gian')
ax.set_ylabel(TARGET_COLUMN)
ax.set_title('Phân chia Train/Test')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ==================== HUẤN LUYỆN SIMPLE ARIMA ====================
if not SIMPLE_ARIMA_AVAILABLE:
    print("❌ Không thể huấn luyện do thiếu SimpleARIMA")
else:
    print("📈 Đang huấn luyện mô hình SimpleARIMA tự viết...")
    print(f"📌 ARIMA order: {ARIMA_ORDER}")
    print(f"📌 Seasonal order: {SEASONAL_ORDER}")
    
    import time
    start_time = time.time()
    
    try:
        # Khởi tạo mô hình SimpleARIMA
        model = SimpleARIMA(
            order=ARIMA_ORDER,
            seasonal_order=SEASONAL_ORDER
        )
        
        # Huấn luyện
        model.fit(y_train)
        training_time = time.time() - start_time
        print(f"\n⏱️ Thời gian huấn luyện: {training_time:.2f} giây")
        
        # Dự báo
        y_pred = model.predict(len(y_test))
        
        # Tạo DataFrame kết quả
        results_df = pd.DataFrame({
            'timestamp': y_test.index,
            'actual': y_test.values,
            'arima_pred': y_pred
        })
        
        print("\n✅ 5 dòng đầu kết quả dự báo:")
        results_df.head(10)
        
    except Exception as e:
        print(f"❌ Lỗi khi huấn luyện SimpleARIMA: {e}")
        import traceback
        traceback.print_exc()
        model = None
        y_pred = np.array([np.nan] * len(y_test))

In [ ]:
# ==================== ĐÁNH GIÁ MÔ HÌNH ====================
if SIMPLE_ARIMA_AVAILABLE and model is not None and not np.isnan(y_pred).all():
    # Tính các metrics
    mae = calculate_mae(y_test.values, y_pred)
    rmse = calculate_rmse(y_test.values, y_pred)
    smape = calculate_smape(y_test.values, y_pred)
    
    print("="*50)
    print("📊 KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH SIMPLE ARIMA")
    print("="*50)
    print(f"✅ MAE  (Mean Absolute Error): {mae:.4f}")
    print(f"✅ RMSE (Root Mean Square Error): {rmse:.4f}")
    print(f"✅ SMAPE (Symmetric MAPE): {smape:.2f}%")
    print("="*50)
    
    # Lưu metrics
    metrics_dict = {
        'model': 'SimpleARIMA',
        'order': str(ARIMA_ORDER),
        'seasonal_order': str(SEASONAL_ORDER),
        'mae': mae,
        'rmse': rmse,
        'smape': smape,
        'training_time': training_time,
        'test_size': len(y_test),
        'train_size': len(y_train)
    }
    
    metrics_df = pd.DataFrame([metrics_dict])
    metrics_df
else:
    print("⚠️ Không thể đánh giá do mô hình không chạy được")
    mae = rmse = smape = np.nan

In [ ]:
# ==================== SO SÁNH VỚI BASELINE ====================
# Đọc kết quả baseline từ notebook 04
baseline_path = os.path.join(project_root, OUTPUT_DIR, 'baseline_metrics.csv')

if os.path.exists(baseline_path) and not np.isnan(mae):
    baseline_metrics = pd.read_csv(baseline_path)
    baseline_mae = baseline_metrics['mae'].values[0]
    baseline_rmse = baseline_metrics['rmse'].values[0]
    baseline_smape = baseline_metrics['smape'].values[0]
    
    print("📊 SO SÁNH SIMPLE ARIMA VỚI BASELINE")
    print("="*60)
    comparison = pd.DataFrame({
        'Metric': ['MAE', 'RMSE', 'SMAPE (%)'],
        'Baseline': [f"{baseline_mae:.4f}", f"{baseline_rmse:.4f}", f"{baseline_smape:.2f}"],
        'SimpleARIMA': [f"{mae:.4f}", f"{rmse:.4f}", f"{smape:.2f}"],
        'Cải thiện': [
            f"{((baseline_mae - mae)/baseline_mae*100):.1f}%",
            f"{((baseline_rmse - rmse)/baseline_rmse*100):.1f}%",
            f"{((baseline_smape - smape)/baseline_smape*100):.1f}%"
        ]
    })
    print(comparison.to_string(index=False))
else:
    print("⚠️ Không tìm thấy kết quả baseline để so sánh")

In [ ]:
# ==================== VẼ BIỂU ĐỒ DỰ BÁO ====================
if PLOT_ARIMA_FORECAST and not np.isnan(y_pred).all():
    # Lấy mẫu 7 ngày đầu của test để dễ nhìn
    sample_days = 7
    sample_points = min(sample_days * 24, len(results_df))
    
    sample_results = results_df.iloc[:sample_points].copy()
    
    fig, ax = plt.subplots(figsize=(15, 6))
    
    ax.plot(sample_results['timestamp'], sample_results['actual'], 
            color='black', linewidth=1.5, label='Thực tế')
    ax.plot(sample_results['timestamp'], sample_results['arima_pred'], 
            color='red', linestyle='--', linewidth=1.5, label='SimpleARIMA')
    
    ax.set_xlabel('Thời gian')
    ax.set_ylabel(TARGET_COLUMN)
    ax.set_title(f'Dự báo SimpleARIMA - {sample_days} ngày đầu tiên của tập test')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.savefig(os.path.join(project_root, 'outputs/figures/simple_arima_forecast.png'), 
                dpi=100, bbox_inches='tight')
    plt.show()
    
    # Vẽ zoom 48 giờ đầu
    fig, ax = plt.subplots(figsize=(15, 5))
    
    zoom_results = results_df.iloc[:min(48, len(results_df))].copy()
    
    ax.plot(zoom_results['timestamp'], zoom_results['actual'], 
            color='black', linewidth=2, marker='o', markersize=4, label='Thực tế')
    ax.plot(zoom_results['timestamp'], zoom_results['arima_pred'], 
            color='red', linestyle='--', linewidth=2, marker='s', markersize=4, label='SimpleARIMA')
    
    ax.set_xlabel('Thời gian')
    ax.set_ylabel(TARGET_COLUMN)
    ax.set_title('Dự báo SimpleARIMA - Zoom 48 giờ đầu')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.savefig(os.path.join(project_root, 'outputs/figures/simple_arima_zoom.png'), 
                dpi=100, bbox_inches='tight')
    plt.show()

In [ ]:
# ==================== PHÂN TÍCH PHẦN DƯ ====================
if not np.isnan(y_pred).all():
    residuals = y_test.values - y_pred
    
    print("📊 Phân tích phần dư (Residuals):")
    print(f"   - Mean: {np.mean(residuals):.4f}")
    print(f"   - Std: {np.std(residuals):.4f}")
    print(f"   - Min: {np.min(residuals):.4f}")
    print(f"   - Max: {np.max(residuals):.4f}")
    
    if abs(np.mean(residuals)) < 0.1:
        print("✅ Residual mean gần 0 - tốt")
    else:
        print("⚠️ Residual mean khác 0 - mô hình có thể bị bias")

In [ ]:
# ==================== VẼ BIỂU ĐỒ PHẦN DƯ ====================
if PLOT_RESIDUALS and not np.isnan(y_pred).all():
    residuals = y_test.values - y_pred
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    
    # 1. Residuals over time
    axes[0,0].plot(y_test.index, residuals, color='purple', linewidth=0.8)
    axes[0,0].axhline(y=0, color='red', linestyle='--', linewidth=1)
    axes[0,0].set_xlabel('Thời gian')
    axes[0,0].set_ylabel('Residuals')
    axes[0,0].set_title('Phần dư theo thời gian')
    axes[0,0].grid(True, alpha=0.3)
    axes[0,0].tick_params(axis='x', rotation=45)
    
    # 2. Histogram of residuals
    axes[0,1].hist(residuals, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
    axes[0,1].axvline(x=0, color='red', linestyle='--', linewidth=2)
    axes[0,1].set_xlabel('Residuals')
    axes[0,1].set_ylabel('Tần suất')
    axes[0,1].set_title('Phân phối phần dư')
    axes[0,1].grid(True, alpha=0.3, axis='y')
    
    # 3. Q-Q plot
    from scipy import stats
    stats.probplot(residuals, dist="norm", plot=axes[1,0])
    axes[1,0].set_title('Q-Q Plot')
    axes[1,0].grid(True, alpha=0.3)
    
    # 4. Actual vs Predicted
    axes[1,1].scatter(y_test.values, y_pred, alpha=0.3, s=1, color='green')
    min_val = min(y_test.min(), y_pred.min())
    max_val = max(y_test.max(), y_pred.max())
    axes[1,1].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2)
    axes[1,1].set_xlabel('Giá trị thực tế')
    axes[1,1].set_ylabel('Giá trị dự báo')
    axes[1,1].set_title('Thực tế vs Dự báo')
    axes[1,1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(project_root, 'outputs/figures/simple_arima_residuals.png'), 
                dpi=100, bbox_inches='tight')
    plt.show()

In [ ]:
# ==================== PHÂN TÍCH LỖI THEO THỜI GIAN ====================
if not np.isnan(y_pred).all():
    residuals = y_test.values - y_pred
    
    # Tạo DataFrame với residuals và thông tin thời gian
    residuals_df = pd.DataFrame({
        'timestamp': y_test.index,
        'actual': y_test.values,
        'predicted': y_pred,
        'residual': residuals,
        'absolute_error': np.abs(residuals),
        'percentage_error': np.abs(residuals) / y_test.values * 100
    })
    
    # Thêm các cột thời gian
    residuals_df['hour'] = residuals_df['timestamp'].dt.hour
    residuals_df['dayofweek'] = residuals_df['timestamp'].dt.dayofweek
    residuals_df['month'] = residuals_df['timestamp'].dt.month
    residuals_df['is_weekend'] = (residuals_df['dayofweek'] >= 5).astype(int)
    
    print("📊 Phân tích lỗi theo giờ:")
    error_by_hour = residuals_df.groupby('hour')['absolute_error'].mean()
    for hour in [0, 6, 12, 18, 23]:
        print(f"   - Giờ {hour}: {error_by_hour[hour]:.4f}")
    
    print("\n📊 Phân tích lỗi theo thứ:")
    dow_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
    error_by_dow = residuals_df.groupby('dayofweek')['absolute_error'].mean()
    for dow in range(7):
        print(f"   - {dow_labels[dow]}: {error_by_dow[dow]:.4f}")

In [ ]:
# ==================== NHẬN XÉT VÀ KẾT LUẬN ====================
if not np.isnan(mae):
    print("="*60)
    print("📝 NHẬN XÉT MÔ HÌNH SIMPLE ARIMA")
    print("="*60)
    
    improvement = ((baseline_mae - mae)/baseline_mae*100) if 'baseline_mae' in locals() else 0
    
    print(f"""
✅ ƯU ĐIỂM:
   - Mô hình ARIMA tự viết, siêu nhẹ (< 5MB)
   - Không cần statsmodels hay pmdarima
   - Đáp ứng đúng yêu cầu đề tài
   - Cải thiện {improvement:.1f}% so với baseline

⚠️ HẠN CHẾ:
   - Phiên bản đơn giản, có thể kém chính xác hơn statsmodels
   - Chưa xử lý phức tạp như ARIMA chuẩn
""")
    
    print(f"\n📊 CHỈ SỐ SIMPLE ARIMA:")
    print(f"   - MAE: {mae:.4f}")
    print(f"   - RMSE: {rmse:.4f}")
    print(f"   - SMAPE: {smape:.2f}%")
    
    print("\n👉 Bước tiếp theo: 06_ets_forecasting.ipynb")

In [ ]:
# ==================== LƯU KẾT QUẢ ====================
# Định nghĩa đường dẫn file
results_path = os.path.join(project_root, OUTPUT_DIR, ARIMA_RESULTS_FILENAME)

if not np.isnan(mae):
    # Lưu kết quả dự báo
    results_df.to_csv(results_path, index=False)
    
    # Lưu metrics
    metrics_path = os.path.join(project_root, OUTPUT_DIR, ARIMA_METRICS_FILENAME)
    metrics_df.to_csv(metrics_path, index=False)
    
    # Lưu residuals
    if 'residuals_df' in locals():
        residuals_df.to_csv(os.path.join(project_root, OUTPUT_DIR, 'simple_arima_residuals.csv'), index=False)
    
    print("\n✅ Đã lưu kết quả thành công:")
    print(f"   - Dự báo: {ARIMA_RESULTS_FILENAME}")
    print(f"   - Metrics: {ARIMA_METRICS_FILENAME}")
else:
    print("\n⚠️ Không lưu kết quả do mô hình không chạy được")
    # Tạo file rỗng để tránh lỗi
    empty_df = pd.DataFrame(columns=['timestamp', 'actual', 'arima_pred'])
    empty_df.to_csv(results_path, index=False)

In [ ]:
# ==================== KIỂM TRA NHANH ====================
if os.path.exists(results_path):
    test_results = pd.read_csv(results_path)
    print(f"✅ Kiểm tra: Đã đọc lại được file {ARIMA_RESULTS_FILENAME}")
    print(f"   - Số dòng: {test_results.shape[0]:,}")
    print(f"   - Các cột: {list(test_results.columns)}")
    
    if test_results.shape[0] > 0 and 'arima_pred' in test_results.columns:
        print("   - 5 dòng đầu:")
        print(test_results.head())
        
        # Kiểm tra không có NaN
        if test_results['arima_pred'].isna().any():
            print("⚠️ CẢNH BÁO: Vẫn còn giá trị NaN trong dự báo!")
            n_nan = test_results['arima_pred'].isna().sum()
            print(f"   - Số lượng NaN: {n_nan}/{len(test_results)} ({n_nan/len(test_results)*100:.1f}%)")
        else:
            print("✅ Tất cả dự báo đều hợp lệ (không NaN)")
    else:
        print("⚠️ File rỗng hoặc không có cột dự báo")
else:
    print(f"⚠️ Không tìm thấy file {results_path}")